# 1. Introduction and research question

This notebook studies whether oil price increases have different effects on financial markets depending on the macroeconomic environment behind the oil move.

The central idea is deliberately simple and transparent. We first build a clean monthly dataset from Bloomberg-style daily and monthly series. We then decompose monthly oil returns into a demand-related part and a residual non-demand-related part, and we test whether this distinction helps predict next-month equity returns and changes in high-yield credit spreads.

Throughout the notebook, the empirical strategy remains reduced-form. The fitted part of the oil return is interpreted as a demand-related oil movement, not as a fully identified structural demand shock. The residual is interpreted as a supply-related or non-demand-related movement, not as a perfectly identified structural supply shock.

# 2. Imports and setup

We keep the environment simple and use only standard libraries that are easy to explain orally: `pandas`, `numpy`, `matplotlib`, `statsmodels`, and `scipy`.

The helper functions are stored in a separate Python file so that the notebook stays readable, but the core empirical logic still appears step by step in the notebook.

In [ ]:
from pathlib import Path
import sys
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.api as sm
from IPython.display import display
from statsmodels.tsa.api import VAR

SRC_PATH = Path("../src")
if not SRC_PATH.exists():
    SRC_PATH = Path("src")
if str(SRC_PATH.resolve()) not in sys.path:
    sys.path.insert(0, str(SRC_PATH.resolve()))

from project_main import (
    DAILY_COLUMNS,
    MONTHLY_COLUMNS,
    add_regime_variables,
    aggregate_daily_to_monthly,
    build_project_variables,
    decompose_oil_returns,
    fit_predictive_regression,
    granger_pvalue_table,
    interpret_two_component_model,
    load_bloomberg_sheet,
    make_summary_statistics,
    normality_check,
    regression_results_table,
    rolling_forecast_comparison,
    run_adf_table,
    show_missing_report,
    split_sample,
)

warnings.filterwarnings("ignore")
plt.style.use("default")
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 140)
pd.set_option("display.float_format", "{:.4f}".format)

DATA_PATH = Path("../data/raw/data_hec_projet_1.xlsx")
if not DATA_PATH.exists():
    DATA_PATH = Path("data/raw/data_hec_projet_1.xlsx")

print("Excel file found:", DATA_PATH.exists())
print("Excel path:", DATA_PATH.resolve())

# 3. Data loading

We start by loading the raw daily and monthly sheets and cleaning the Bloomberg header rows. At this stage we do not transform the data yet. The goal is simply to verify that the dates, variable names, and sample ranges are coherent.

In [ ]:
daily_raw = load_bloomberg_sheet(DATA_PATH, "Daily", DAILY_COLUMNS)
monthly_raw = load_bloomberg_sheet(DATA_PATH, "Monthly", MONTHLY_COLUMNS)

print("Daily raw shape:", daily_raw.shape)
print("Monthly raw shape:", monthly_raw.shape)
print("Daily date range:", daily_raw["date"].min(), "to", daily_raw["date"].max())
print("Monthly date range:", monthly_raw["date"].min(), "to", monthly_raw["date"].max())

print("\nCheck that the raw datasets are sorted and have unique monthly dates where expected:")
checks = {
    "Daily dates are sorted": daily_raw["date"].is_monotonic_increasing,
    "Monthly dates are sorted": monthly_raw["date"].is_monotonic_increasing,
    "Monthly dates are unique": monthly_raw["date"].is_unique,
}
for label, value in checks.items():
    print(f"{label}: {value}")

print("\nTable 1. First rows of the cleaned daily dataset")
display(daily_raw.head())

print("Table 2. First rows of the cleaned monthly macro dataset")
display(monthly_raw.head())

# 4. Data cleaning and monthly aggregation

The core analysis is monthly, so we aggregate the daily market data to month-end using the last available observation of each month. This is a natural choice for market prices and yields because it keeps the monthly panel aligned with the macro series.

We also compute realized monthly S&P 500 volatility from daily log returns inside each month. Before moving on, we explicitly inspect missing values before and after the merge.

In [ ]:
daily_monthly = aggregate_daily_to_monthly(daily_raw)

print("Aggregated daily-to-monthly shape:", daily_monthly.shape)
print("Monthly aggregated date range:", daily_monthly["date"].min(), "to", daily_monthly["date"].max())

_ = show_missing_report(daily_monthly, "Aggregated daily data before merge")
_ = show_missing_report(monthly_raw, "Monthly macro data before merge")

monthly_merged = (
    daily_monthly.merge(monthly_raw, on="date", how="inner")
    .sort_values("date")
    .reset_index(drop=True)
)

print("\nMerged monthly dataset shape:", monthly_merged.shape)
print("Merged date range:", monthly_merged["date"].min(), "to", monthly_merged["date"].max())
_ = show_missing_report(monthly_merged, "Merged monthly dataset before variable construction")

print("\nVerification loop for the merged monthly panel:")
merged_checks = {
    "One row per month": monthly_merged["date"].is_unique,
    "Dates remain sorted": monthly_merged["date"].is_monotonic_increasing,
    "Realized volatility column exists": "sp500_realized_vol" in monthly_merged.columns,
    "No duplicate dates": monthly_merged["date"].duplicated().sum() == 0,
}
for label, value in merged_checks.items():
    print(f"{label}: {value}")

print("\nTable 3. First rows of the merged monthly dataset")
display(monthly_merged.head())

print("Table 4. Last rows of the merged monthly dataset")
display(monthly_merged.tail())

# 5. Variable construction

We now construct the main variables used in the empirical analysis.

Monthly returns are computed in logs because log returns are additive over time and are standard in empirical finance for market prices. The term spread is kept as a level difference because it is already an economically meaningful spread between long and short rates. High-yield yield-to-worst is used in monthly change because the project focuses on widening or tightening in credit conditions rather than on the level alone.

In [ ]:
project_df = build_project_variables(monthly_merged)

print("Project dataset shape after variable construction:", project_df.shape)

core_columns = [
    "date",
    "wti_return",
    "brent_return",
    "sp500_return",
    "msci_em_return",
    "msci_world_return",
    "russell2000_return",
    "gold_return",
    "term_spread",
    "hy_change",
    "sp500_realized_vol",
    "cfnai",
    "ism_mfg",
    "ip_growth",
    "retail_sales_growth",
]

print("\nVerification loop for core project variables:")
for column in core_columns[1:]:
    first_valid = project_df[column].first_valid_index()
    last_valid = project_df[column].last_valid_index()
    print(f"{column}: first valid index = {first_valid}, last valid index = {last_valid}")

print("\nTable 5. Preview of the main constructed variables")
display(project_df[core_columns].head(12))

# 6. Descriptive statistics and stationarity

Before estimating predictive models, it is useful to understand the distribution of the main variables, their comovement, and whether the basic transformations appear sensible.

We keep this section descriptive and transparent: summary statistics, correlations, a few simple plots, and ADF tests for the main monthly series. Phillips-Perron is omitted for package simplicity, which we state clearly rather than adding an extra dependency.

In [ ]:
descriptive_columns = [
    "wti_return",
    "sp500_return",
    "hy_change",
    "term_spread",
    "gold_return",
    "sp500_realized_vol",
    "cfnai",
    "ism_mfg",
    "ip_growth",
    "retail_sales_growth",
]

print("Table 6. Summary statistics for the main variables")
summary_table = make_summary_statistics(project_df, descriptive_columns)
display(summary_table)

print("Table 7. Correlation matrix for the main variables")
correlation_table = project_df[descriptive_columns].corr().round(3)
display(correlation_table)

normality_rows = []
for column in ["wti_return", "sp500_return", "hy_change"]:
    result = normality_check(project_df[column])
    normality_rows.append(
        {
            "variable": column,
            "jarque_bera_stat": result["statistic"],
            "p_value": result["p_value"],
        }
    )

print("Table 8. Simple normality checks for selected financial variables")
display(pd.DataFrame(normality_rows))

In [ ]:
fig, axes = plt.subplots(5, 1, figsize=(12, 14), sharex=True)

plot_columns = [
    ("wti_return", "Figure 1. Monthly WTI log return"),
    ("sp500_return", "Figure 2. Monthly S&P 500 log return"),
    ("hy_change", "Figure 3. Monthly change in HY yield-to-worst"),
    ("term_spread", "Figure 4. Term spread (10Y minus 2Y)"),
    ("cfnai", "Figure 5. CFNAI"),
]

for ax, (column, title) in zip(axes, plot_columns):
    ax.plot(project_df["date"], project_df[column])
    ax.set_title(title)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(project_df["wti_return"].dropna(), bins=30)
axes[0].set_title("Figure 6. Histogram of monthly WTI log returns")
axes[1].hist(project_df["sp500_return"].dropna(), bins=30)
axes[1].set_title("Figure 7. Histogram of monthly S&P 500 log returns")
plt.tight_layout()
plt.show()

In [ ]:
stationarity_columns = [
    "wti_return",
    "sp500_return",
    "hy_change",
    "term_spread",
    "gold_return",
    "cfnai",
    "ism_mfg",
]

print("Table 9. Augmented Dickey-Fuller tests")
adf_table = run_adf_table(project_df, stationarity_columns)
display(adf_table)

print("Phillips-Perron test is omitted here to keep the notebook limited to a simple standard package set.")

print("\nShort interpretation of the ADF results:")
for row in adf_table.itertuples(index=False):
    if pd.isna(row.p_value):
        print(f"{row.variable}: not enough data to run the test cleanly.")
    elif row.stationary_5pct:
        print(f"{row.variable}: appears stationary at the 5% level, so the transformation looks reasonable.")
    else:
        print(f"{row.variable}: does not clearly reject a unit root at the 5% level, so interpretation should stay cautious.")

# 7. Oil shock decomposition

This is the key part of the project. We implement two complementary reduced-form approaches.

In the baseline specification, monthly WTI returns are regressed on CFNAI. The fitted value is interpreted as the demand-related component of oil movements, while the residual is interpreted as the non-demand-related component. This is not a structural identification strategy, but it provides a transparent way to separate oil moves that coincide with stronger macro activity from oil moves that do not.

As a robustness exercise, we repeat the same idea with ISM Manufacturing. We also build a simpler regime-based specification that interacts oil returns with expansion and contraction periods defined by the ISM threshold of 50.

In [ ]:
decomposition_df, cfnai_decomp_model = decompose_oil_returns(
    project_df,
    oil_return_col="wti_return",
    activity_col="cfnai",
    prefix="baseline",
)

decomposition_df, ism_decomp_model = decompose_oil_returns(
    decomposition_df,
    oil_return_col="wti_return",
    activity_col="ism_mfg",
    prefix="ism",
)

decomposition_df = add_regime_variables(
    decomposition_df,
    oil_col="wti_return",
    ism_col="ism_mfg",
)

print("Table 10. Regression-based decomposition using CFNAI")
display(regression_results_table(cfnai_decomp_model))

print("Table 11. Regression-based decomposition using ISM Manufacturing")
display(regression_results_table(ism_decomp_model))

reconstruction_gap = (
    decomposition_df["wti_return"]
    - decomposition_df["baseline_oil_demand_component"]
    - decomposition_df["baseline_oil_supply_component"]
).dropna()
print("Maximum absolute reconstruction gap:", reconstruction_gap.abs().max())

print("\nReminder: both components are reduced-form objects, not structural shocks.")

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)

axes[0].plot(decomposition_df["date"], decomposition_df["wti_return"], label="WTI return")
axes[0].plot(
    decomposition_df["date"],
    decomposition_df["baseline_oil_demand_component"],
    label="Demand-related component",
)
axes[0].set_title("Figure 8. WTI return and demand-related component")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(
    decomposition_df["date"],
    decomposition_df["baseline_oil_supply_component"],
    label="Supply-related residual",
)
axes[1].set_title("Figure 9. Supply-related or non-demand-related residual")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

axes[2].plot(decomposition_df["date"], decomposition_df["oil_expansion"], label="Oil x expansion")
axes[2].plot(decomposition_df["date"], decomposition_df["oil_contraction"], label="Oil x contraction")
axes[2].set_title("Figure 10. Regime-based oil variables")
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# 8. Predictive regressions

We now estimate one-month-ahead predictive regressions. The key timing choice is important: predictors are measured at time *t* and the dependent variable is shifted to *t+1*, so the regressions avoid look-ahead bias.

The baseline specification uses the demand-related and supply-related oil components, together with a small control set: the current value of the dependent variable, the term spread, gold return, and realized S&P 500 volatility. We estimate the regressions with heteroskedasticity-robust standard errors.

In [ ]:
def interpret_pair(model, variables, labels):
    for variable, label in zip(variables, labels):
        coef = model.params.get(variable, np.nan)
        p_value = model.pvalues.get(variable, np.nan)
        sign = "positive" if coef > 0 else "negative"
        significance = "statistically significant" if p_value < 0.05 else "not statistically significant"
        print(f"{label}: coefficient is {sign} ({coef:.4f}) and {significance} at the 5% level (p-value = {p_value:.4f}).")

target_map = {
    "sp500_return": "S&P 500 next-month return",
    "hy_change": "High-yield next-month change",
}

decomposition_models = {}
regime_models = {}

for target, label in target_map.items():
    decomposition_predictors = [
        "baseline_oil_demand_component",
        "baseline_oil_supply_component",
        target,
        "term_spread",
        "gold_return",
        "sp500_realized_vol",
    ]

    decomp_model, decomp_sample = fit_predictive_regression(
        decomposition_df,
        dependent_col=target,
        predictor_cols=decomposition_predictors,
        horizon=1,
        cov_type="HC1",
    )
    decomposition_models[target] = (decomp_model, decomp_sample)

    print(f"Table for {label}: decomposition model")
    print("Regression sample size:", len(decomp_sample))
    display(regression_results_table(decomp_model))
    interpret_pair(
        decomp_model,
        ["baseline_oil_demand_component", "baseline_oil_supply_component"],
        ["Demand-related oil effect", "Supply-related oil effect"],
    )
    print("")

    regime_predictors = [
        "oil_expansion",
        "oil_contraction",
        target,
        "term_spread",
        "gold_return",
        "sp500_realized_vol",
    ]

    regime_model, regime_sample = fit_predictive_regression(
        decomposition_df,
        dependent_col=target,
        predictor_cols=regime_predictors,
        horizon=1,
        cov_type="HC1",
    )
    regime_models[target] = (regime_model, regime_sample)

    print(f"Table for {label}: regime-based model")
    print("Regression sample size:", len(regime_sample))
    display(regression_results_table(regime_model))
    interpret_pair(
        regime_model,
        ["oil_expansion", "oil_contraction"],
        ["Oil effect during ISM expansion", "Oil effect during ISM contraction"],
    )
    print("-" * 100)

# 9. Granger causality tests

The Granger causality exercise is kept intentionally narrow. Instead of testing many combinations mechanically, we focus on the four relationships most directly linked to the project question: whether the two oil components help predict future equity returns and future changes in high-yield spreads.

In [ ]:
granger_specs = [
    ("baseline_oil_demand_component", "sp500_return"),
    ("baseline_oil_supply_component", "sp500_return"),
    ("baseline_oil_demand_component", "hy_change"),
    ("baseline_oil_supply_component", "hy_change"),
]

granger_tables = []
for cause, effect in granger_specs:
    table = granger_pvalue_table(decomposition_df, cause, effect, max_lag=3)
    granger_tables.append(table)

granger_summary = pd.concat(granger_tables, ignore_index=True)
print("Table 12. Granger causality p-values")
display(granger_summary)

print("\nShort reading of the Granger results:")
for (cause, effect), group in granger_summary.groupby(["cause", "effect"]):
    best_p = group["p_value"].min()
    if best_p < 0.05:
        print(f"{cause} shows some predictive content for {effect} over at least one lag in the 1 to 3 range.")
    else:
        print(f"{cause} does not show strong Granger evidence for {effect} over lags 1 to 3.")

# 10. VAR and impulse responses

As an extension, we estimate a reduced-form monthly VAR. The goal is not structural identification. Instead, the VAR gives a compact way to inspect dynamic comovement and to see whether the impulse responses tell a story broadly consistent with the predictive regressions.

In [ ]:
var_columns = [
    "wti_return",
    "sp500_return",
    "term_spread",
    "hy_change",
    "gold_return",
    "ism_mfg",
]

var_df = decomposition_df[["date"] + var_columns].dropna().copy()
var_df = var_df.set_index("date")

print("VAR sample shape:", var_df.shape)
print("VAR sample period:", var_df.index.min(), "to", var_df.index.max())

var_model = VAR(var_df)
lag_selection = var_model.select_order(maxlags=6)

lag_choice_table = pd.DataFrame([lag_selection.selected_orders], index=["selected_lag"])
print("Table 13. VAR lag choice suggested by information criteria")
display(lag_choice_table)

chosen_lag = lag_selection.selected_orders["bic"]
if chosen_lag == 0:
    chosen_lag = max(1, lag_selection.selected_orders["aic"])

print("Chosen lag for the baseline VAR:", chosen_lag)
var_results = var_model.fit(chosen_lag)

print("Table 14. Reduced-form VAR coefficients")
display(var_results.params.round(4))

print("Table 15. Reduced-form VAR p-values")
display(var_results.pvalues.round(4))

In [ ]:
irf_horizon = 18
irf = var_results.irf(irf_horizon)

fig1 = irf.plot(impulse="wti_return", response="sp500_return")
fig1.set_size_inches(8, 5)
fig1.axes[0].set_title("Figure 11. IRF of S&P 500 return to a WTI return shock")
plt.tight_layout()
plt.show()

fig2 = irf.plot(impulse="wti_return", response="hy_change")
fig2.set_size_inches(8, 5)
fig2.axes[0].set_title("Figure 12. IRF of HY change to a WTI return shock")
plt.tight_layout()
plt.show()

fig3 = irf.plot(impulse="wti_return", response="term_spread")
fig3.set_size_inches(8, 5)
fig3.axes[0].set_title("Figure 13. IRF of term spread to a WTI return shock")
plt.tight_layout()
plt.show()

try:
    fevd = var_results.fevd(12)
    shock_index = list(var_df.columns).index("wti_return")
    fevd_rows = []
    for variable in ["sp500_return", "hy_change", "term_spread"]:
        variable_index = list(var_df.columns).index(variable)
        share = fevd.decomp[variable_index, 11, shock_index]
        fevd_rows.append(
            {
                "variable": variable,
                "share_of_12_month_fevd_explained_by_wti_shock": share,
            }
        )

    print("Table 16. Simple FEVD summary at the 12-month horizon")
    display(pd.DataFrame(fevd_rows).round(4))
except Exception as error:
    print("FEVD is omitted because the simple summary extraction failed:", error)

# 11. Out-of-sample exercise

We now compare three simple forecasting approaches in an expanding-window setting:

1. A historical mean benchmark.
2. A model with the raw oil return.
3. A model with the demand-related and supply-related oil components.

This part is intentionally modest. The goal is simply to see whether the decomposition adds predictive value relative to a benchmark that treats oil as a single undifferentiated return series.

In [ ]:
forecast_targets = {
    "sp500_return": "S&P 500 next-month return",
    "hy_change": "High-yield next-month change",
}

forecast_metrics = {}

for target, label in forecast_targets.items():
    forecast_df, metrics_df = rolling_forecast_comparison(
        decomposition_df,
        target_col=target,
        raw_oil_col="wti_return",
        demand_col="baseline_oil_demand_component",
        supply_col="baseline_oil_supply_component",
        controls=["term_spread", "gold_return", "sp500_realized_vol"],
        start_share=0.6,
    )

    forecast_metrics[target] = (forecast_df, metrics_df)
    print(f"Table for {label}: out-of-sample forecasting performance")
    display(metrics_df)

    if not forecast_df.empty:
        fig, ax = plt.subplots(figsize=(10, 4))
        plot_df = forecast_df.set_index("date")
        plot_df[["actual", "benchmark", "raw_oil_model", "decomposition_model"]].tail(60).plot(ax=ax)
        ax.set_title(f"Figure. Recent forecasts for {label}")
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()

# 12. Robustness checks

We keep the robustness section short and focused. The aim is not to multiply specifications, but to check whether the core message survives a few sensible variations:

- replacing WTI with Brent,
- using ISM rather than CFNAI in the decomposition,
- changing the equity market from the S&P 500 to MSCI Emerging Markets,
- and splitting the sample around the 2008 financial crisis.

In [ ]:
robustness_rows = []

brent_df, brent_decomp_model = decompose_oil_returns(
    decomposition_df,
    oil_return_col="brent_return",
    activity_col="cfnai",
    prefix="brent",
)
brent_predictive_model, brent_sample = fit_predictive_regression(
    brent_df,
    dependent_col="sp500_return",
    predictor_cols=[
        "brent_oil_demand_component",
        "brent_oil_supply_component",
        "sp500_return",
        "term_spread",
        "gold_return",
        "sp500_realized_vol",
    ],
    horizon=1,
    cov_type="HC1",
)
robustness_rows.append(
    {
        "check": "Brent instead of WTI",
        "sample_size": len(brent_sample),
        "component_1": brent_predictive_model.params.get("brent_oil_demand_component", np.nan),
        "component_1_pvalue": brent_predictive_model.pvalues.get("brent_oil_demand_component", np.nan),
        "component_2": brent_predictive_model.params.get("brent_oil_supply_component", np.nan),
        "component_2_pvalue": brent_predictive_model.pvalues.get("brent_oil_supply_component", np.nan),
    }
)

ism_predictive_model, ism_sample = fit_predictive_regression(
    decomposition_df,
    dependent_col="sp500_return",
    predictor_cols=[
        "ism_oil_demand_component",
        "ism_oil_supply_component",
        "sp500_return",
        "term_spread",
        "gold_return",
        "sp500_realized_vol",
    ],
    horizon=1,
    cov_type="HC1",
)
robustness_rows.append(
    {
        "check": "ISM decomposition instead of CFNAI",
        "sample_size": len(ism_sample),
        "component_1": ism_predictive_model.params.get("ism_oil_demand_component", np.nan),
        "component_1_pvalue": ism_predictive_model.pvalues.get("ism_oil_demand_component", np.nan),
        "component_2": ism_predictive_model.params.get("ism_oil_supply_component", np.nan),
        "component_2_pvalue": ism_predictive_model.pvalues.get("ism_oil_supply_component", np.nan),
    }
)

em_predictive_model, em_sample = fit_predictive_regression(
    decomposition_df,
    dependent_col="msci_em_return",
    predictor_cols=[
        "baseline_oil_demand_component",
        "baseline_oil_supply_component",
        "msci_em_return",
        "term_spread",
        "gold_return",
        "sp500_realized_vol",
    ],
    horizon=1,
    cov_type="HC1",
)
robustness_rows.append(
    {
        "check": "MSCI EM instead of S&P 500",
        "sample_size": len(em_sample),
        "component_1": em_predictive_model.params.get("baseline_oil_demand_component", np.nan),
        "component_1_pvalue": em_predictive_model.pvalues.get("baseline_oil_demand_component", np.nan),
        "component_2": em_predictive_model.params.get("baseline_oil_supply_component", np.nan),
        "component_2_pvalue": em_predictive_model.pvalues.get("baseline_oil_supply_component", np.nan),
    }
)

pre_2008, post_2008 = split_sample(decomposition_df, pd.Timestamp("2008-09-30"))

if len(pre_2008) > 60:
    pre_model, pre_sample = fit_predictive_regression(
        pre_2008,
        dependent_col="sp500_return",
        predictor_cols=[
            "baseline_oil_demand_component",
            "baseline_oil_supply_component",
            "sp500_return",
            "term_spread",
            "gold_return",
            "sp500_realized_vol",
        ],
        horizon=1,
        cov_type="HC1",
    )
    robustness_rows.append(
        {
            "check": "Pre-2008 sample",
            "sample_size": len(pre_sample),
            "component_1": pre_model.params.get("baseline_oil_demand_component", np.nan),
            "component_1_pvalue": pre_model.pvalues.get("baseline_oil_demand_component", np.nan),
            "component_2": pre_model.params.get("baseline_oil_supply_component", np.nan),
            "component_2_pvalue": pre_model.pvalues.get("baseline_oil_supply_component", np.nan),
        }
    )

if len(post_2008) > 60:
    post_model, post_sample = fit_predictive_regression(
        post_2008,
        dependent_col="sp500_return",
        predictor_cols=[
            "baseline_oil_demand_component",
            "baseline_oil_supply_component",
            "sp500_return",
            "term_spread",
            "gold_return",
            "sp500_realized_vol",
        ],
        horizon=1,
        cov_type="HC1",
    )
    robustness_rows.append(
        {
            "check": "Post-2008 sample",
            "sample_size": len(post_sample),
            "component_1": post_model.params.get("baseline_oil_demand_component", np.nan),
            "component_1_pvalue": post_model.pvalues.get("baseline_oil_demand_component", np.nan),
            "component_2": post_model.params.get("baseline_oil_supply_component", np.nan),
            "component_2_pvalue": post_model.pvalues.get("baseline_oil_supply_component", np.nan),
        }
    )

print("Table 17. Selected robustness checks")
display(pd.DataFrame(robustness_rows).round(4))

# 13. Main conclusions

The goal of the project is not to claim a fully structural causal interpretation. Instead, the contribution of the notebook is to show, in a transparent monthly framework, whether it is empirically useful to distinguish oil price increases that coincide with stronger activity from oil price increases that do not.

The final code block below turns the main outputs into a short project-style conclusion. This keeps the interpretation tied to the actual regression and forecasting results obtained in the notebook.

In [ ]:
sp500_model = decomposition_models["sp500_return"][0]
hy_model = decomposition_models["hy_change"][0]

sp500_forecast_metrics = forecast_metrics["sp500_return"][1].set_index("model")
hy_forecast_metrics = forecast_metrics["hy_change"][1].set_index("model")

print("Project conclusion summary")
print("")
print("1. The decomposition remains reduced-form throughout the notebook.")
print("   The fitted part is interpreted as demand-related and the residual as supply-related or non-demand-related.")
print("")
print("2. Predictive regressions for next-month S&P 500 returns:")
for line in interpret_two_component_model(
    sp500_model,
    "baseline_oil_demand_component",
    "baseline_oil_supply_component",
):
    print("   " + line)
print("")
print("3. Predictive regressions for next-month HY changes:")
for line in interpret_two_component_model(
    hy_model,
    "baseline_oil_demand_component",
    "baseline_oil_supply_component",
):
    print("   " + line)
print("")
print("4. Out-of-sample forecasting for S&P 500 returns:")
print(sp500_forecast_metrics)
print("")
print("5. Out-of-sample forecasting for HY changes:")
print(hy_forecast_metrics)
print("")
print("These results should be read as transparent empirical evidence, not as full causal identification.")